# Step 4 of RAG: Passing Context to the Gemini / Groq LLM for Answering**RAG** (Retrieval-Augmented Generation) pipeline recap:1. Document Loading & Chunking (Step 1)2. Storing chunks in a Vector Database (Step 2)3. Retrieving relevant context for a user query (Step 3)4. **Passing the retrieved context to an LLM to generate the final answer** ← *this notebook*We use two **free-tier** LLM providers (you can use either one, or both):| Provider | Free API key from | Model family ||---|---|---|| **Groq** | https://console.groq.com/keys | Llama / GPT-OSS (very fast) || **Google Gemini** | https://aistudio.google.com/apikey | Gemini Flash |> **You only need ONE key.** If you skip a provider, the notebook simply skips that part — no errors.> **Security tip:** never paste API keys directly into notebook code or share a notebook with a key inside it.### What you will learn1. Safely entering API keys in Colab2. Rebuilding the retrieval pipeline (Steps 1-3) in one condensed block3. Writing a **grounded system prompt** that stops hallucination4. Calling **Groq** with the retrieved context5. Calling **Gemini** with the retrieved context6. Building one `rag_answer()` function that works with either provider7. Testing: in-scope question, multi-chunk question, and out-of-scope question8. Comparing both LLMs on the same question

### Setup — Install Required Libraries

In [1]:
!pip install -q langchain-text-splitters chromadb sentence-transformers groq google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.

## 1. Enter Your API Keys SafelyThis cell looks for a key in three places, in order:1. An environment variable2. Colab **Secrets** (the key icon in the left sidebar) — the safest option3. A hidden input box (`getpass`) — just press **Enter** to skip a providerThe names to use are `GROQ_API_KEY` and `GEMINI_API_KEY`.

In [2]:
import os
import getpass

def get_key(name):
    key = os.environ.get(name, "")
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get(name) or ""
        except Exception:
            key = ""
    if not key:
        key = getpass.getpass(f"Enter {name} (press Enter to skip): ").strip()
    return key

GROQ_API_KEY = get_key("GROQ_API_KEY")
GEMINI_API_KEY = get_key("GEMINI_API_KEY")

print("Groq key provided:  ", bool(GROQ_API_KEY))
print("Gemini key provided:", bool(GEMINI_API_KEY))

if not GROQ_API_KEY and not GEMINI_API_KEY:
    print("\nNo keys provided. The retrieval parts will still run, but LLM answers will be skipped.")


Enter GROQ_API_KEY (press Enter to skip): ··········
Enter GEMINI_API_KEY (press Enter to skip): ··········
Groq key provided:   False
Gemini key provided: False

No keys provided. The retrieval parts will still run, but LLM answers will be skipped.


## 2. Recap — Chunk, Store and Retrieve (Steps 1-3 Condensed)Same pipeline as the earlier notebooks, condensed so this notebook runs on its own.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions

documents = {
    "ai_overview.txt": """Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems that can perform tasks which normally require human intelligence. These tasks include understanding language, recognizing images, making decisions, and learning from experience.

Machine Learning is a subset of AI where systems learn patterns from data instead of following hardcoded rules. Instead of programming every decision by hand, we feed the system many examples, and it learns the underlying pattern on its own.

Deep Learning takes Machine Learning further by using neural networks with many layers. These layered networks can automatically discover complex patterns in large amounts of data, such as recognizing faces in photos or understanding the meaning of a sentence.

Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a language model. Instead of relying only on what a language model memorized during training, RAG first retrieves relevant, up-to-date information from a document collection, then passes that  information to the model so it can generate a more accurate, grounded answer.""",
    "embeddings_in_practice.txt": """Embeddings and Vector Search in Practice

Modern search engines increasingly rely on embeddings rather than simple keyword matching. An embedding model converts a search query into the same vector space as the stored documents, allowing the system to find results that match the meaning of the query, not just its exact words. This is especially powerful for customer support systems, where a user might phrase a question very differently from how it appears in a help article, yet still expect to find the right answer.

Vector databases such as ChromaDB store these embeddings and allow fast similarity search across thousands or millions of documents at once."""
}

splitter = RecursiveCharacterTextSplitter(
    chunk_size=250, chunk_overlap=40, separators=["\n\n", "\n", ". ", " ", ""]
)

client = chromadb.Client()
hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

try:
    client.delete_collection("rag_chunks")
except Exception:
    pass

collection = client.create_collection(
    name="rag_chunks", embedding_function=hf_embedder
)

for source, text in documents.items():
    doc_chunks = splitter.split_text(text)
    collection.add(
        documents=doc_chunks,
        metadatas=[{"source": source, "chunk_index": i} for i in range(len(doc_chunks))],
        ids=[f"{source}_chunk_{i}" for i in range(len(doc_chunks))]
    )

print("Total chunks stored:", collection.count())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total chunks stored: 14


In [4]:
def retrieve_context(query, n_results=4, similarity_threshold=0.25):
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )
    context_chunks = []
    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        similarity = 1 - distance
        if similarity >= similarity_threshold:
            context_chunks.append({
                "text": doc,
                "source": meta["source"],
                "chunk_index": meta["chunk_index"],
                "similarity": round(similarity, 4)
            })
    return context_chunks

# Quick check that retrieval works
test_chunks = retrieve_context("What is RAG?")
print("Chunks retrieved:", len(test_chunks))
for c in test_chunks:
    print(f"  [{c['similarity']}] {c['source']} #{c['chunk_index']}")


Chunks retrieved: 1
  [0.4284] ai_overview.txt #6


## 3. Writing a Grounded PromptThe prompt is what turns a general-purpose LLM into a **RAG** system. Two parts matter:- **System prompt** — the rules: *answer only from the context, say "I don't know" otherwise, cite sources.*- **User prompt** — the retrieved context + the user's question.Setting a **low temperature** (0.1) makes the model stick to the facts instead of being creative.

In [5]:
SYSTEM_PROMPT = (
    "You are a helpful assistant that answers questions using ONLY the provided context. "
    "If the answer is not in the context, reply exactly: "
    "\"I don't know based on the provided documents.\" "
    "Keep the answer short and clear. At the end, mention the source file name(s) you used."
)

def build_user_prompt(query, context_chunks):
    context_text = "\n\n".join(
        f"[Source: {c['source']}, chunk {c['chunk_index']}]\n{c['text']}"
        for c in context_chunks
    )
    return f"Context:\n{context_text}\n\nQuestion: {query}\n\nAnswer:"

sample_chunks = retrieve_context("How does RAG improve a language model?")
print(build_user_prompt("How does RAG improve a language model?", sample_chunks))


Context:
[Source: ai_overview.txt, chunk 7]
. Instead of relying only on what a language model memorized during training, RAG first retrieves relevant, up-to-date information from a document collection, then passes that  information to the model so it can generate a more accurate, grounded

[Source: ai_overview.txt, chunk 6]
Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a language model

[Source: ai_overview.txt, chunk 2]
. These tasks include understanding language, recognizing images, making decisions, and learning from experience.

[Source: embeddings_in_practice.txt, chunk 2]
. An embedding model converts a search query into the same vector space as the stored documents, allowing the system to find results that match the meaning of the query, not just its exact words

Question: How does RAG improve a language model?

Answer:


**Discussion Point:** The LLM never "searches" anything itself. It only sees the text we paste into theprompt — so the quality of the answer depends directly on the quality of the retrieved chunks.

## 4. Calling Groq (Llama / GPT-OSS models)Groq exposes an OpenAI-style chat API. We try a short list of models in order, so if one isrenamed or temporarily unavailable, the next one is used automatically instead of crashing.

In [6]:
from groq import Groq

GROQ_MODELS = [
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant",
    "openai/gpt-oss-20b"
]

def ask_groq(user_prompt):
    if not GROQ_API_KEY:
        return "[Groq skipped: no API key provided]"
    groq_client = Groq(api_key=GROQ_API_KEY)
    last_error = None

    for model_name in GROQ_MODELS:
        try:
            response = groq_client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.1,
                max_tokens=600
            )
            answer = response.choices[0].message.content
            if answer:
                return answer.strip()
        except Exception as e:
            last_error = e

    return f"[Groq error: {last_error}]"

question = "How does RAG improve a language model?"
chunks = retrieve_context(question)
print("Groq answer:\n")
print(ask_groq(build_user_prompt(question, chunks)))


Groq answer:

[Groq skipped: no API key provided]


## 5. Calling Google GeminiWe use the official `google-genai` SDK. Again, a short model list gives us an automatic fallback.`system_instruction` plays the same role as the system message we used with Groq.

In [7]:
from google import genai
from google.genai import types

GEMINI_MODELS = [
    "gemini-2.5-flash",
    "gemini-3-flash-preview",
    "gemini-2.5-flash-lite"
]

def ask_gemini(user_prompt):
    if not GEMINI_API_KEY:
        return "[Gemini skipped: no API key provided]"
    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    last_error = None

    for model_name in GEMINI_MODELS:
        try:
            response = gemini_client.models.generate_content(
                model=model_name,
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    temperature=0.1,
                    max_output_tokens=1500
                )
            )
            if response.text:
                return response.text.strip()
        except Exception as e:
            last_error = e

    return f"[Gemini error: {last_error}]"

print("Gemini answer:\n")
print(ask_gemini(build_user_prompt(question, chunks)))


Gemini answer:

[Gemini skipped: no API key provided]


## 6. One Function for the Full RAG Pipeline: `rag_answer()`This ties everything together: **retrieve → build prompt → ask the LLM → show sources**.If retrieval finds nothing relevant, we skip the LLM completely — that saves tokens and avoids amade-up answer.

In [8]:
def rag_answer(query, provider="groq", show_sources=True):
    context_chunks = retrieve_context(query)

    if not context_chunks:
        return "I don't know based on the provided documents. (No relevant chunks were retrieved.)"

    user_prompt = build_user_prompt(query, context_chunks)

    if provider == "gemini":
        answer = ask_gemini(user_prompt)
    else:
        answer = ask_groq(user_prompt)

    if show_sources:
        sources = sorted({
            f"{c['source']} #{c['chunk_index']}"
            for c in context_chunks
        })
        answer += "\n\nRetrieved from: " + ", ".join(sources)

    return answer

default_provider = "groq" if GROQ_API_KEY else "gemini"
print("Using provider:", default_provider)


Using provider: gemini


## 7. Testing the Full RAG Pipeline

**Test 1 — A simple in-scope question**

In [9]:
q1 = "What is deep learning?"
print("Question:", q1, "\n")
print(rag_answer(q1, provider=default_provider))

Question: What is deep learning? 

[Gemini skipped: no API key provided]

Retrieved from: ai_overview.txt #1, ai_overview.txt #3, ai_overview.txt #4, ai_overview.txt #5


**Test 2 — A question that needs information from more than one document**

In [10]:
q2 = "How do embeddings and vector databases help RAG find useful information?"
print("Question:", q2, "\n")
print(rag_answer(q2, provider=default_provider))

Question: How do embeddings and vector databases help RAG find useful information? 

[Gemini skipped: no API key provided]

Retrieved from: embeddings_in_practice.txt #0, embeddings_in_practice.txt #1, embeddings_in_practice.txt #2, embeddings_in_practice.txt #4


**Test 3 — An out-of-scope question**Our documents say nothing about cooking. A good RAG system must admit that instead of guessing.

In [11]:
q3 = "What is the best recipe for chocolate cake?"
print("Question:", q3, "\n")
print(rag_answer(q3, provider=default_provider))

Question: What is the best recipe for chocolate cake? 

I don't know based on the provided documents. (No relevant chunks were retrieved.)


**Discussion Point:** Test 3 is the most important one. Without the similarity threshold and thegrounded system prompt, a plain LLM would happily invent a cake recipe. RAG's job is to stayinside the knowledge base.

## 8. Comparing Groq and Gemini on the Same QuestionIf you provided both keys, this cell shows both answers side by side. If you gave only one key,the other provider is politely skipped.

In [12]:
compare_q = "Why is RAG more reliable than asking a language model directly?"
print("Question:", compare_q, "\n")

compare_chunks = retrieve_context(compare_q)
compare_prompt = build_user_prompt(compare_q, compare_chunks)

print("=" * 25, "GROQ", "=" * 25)
print(ask_groq(compare_prompt))

print("\n" + "=" * 24, "GEMINI", "=" * 24)
print(ask_gemini(compare_prompt))

Question: Why is RAG more reliable than asking a language model directly? 

========================= GROQ =========================
[Groq skipped: no API key provided]

======================== GEMINI ========================
[Gemini skipped: no API key provided]


## 9. Interactive Q&A (Optional)Ask your own questions about the documents. Leave the box empty and press Enter to stop.

In [13]:
while True:
    user_question = input("Ask a question (or press Enter to quit): ").strip()
    if not user_question:
        print("Goodbye!")
        break
    print("\n" + rag_answer(user_question, provider=default_provider) + "\n")

Ask a question (or press Enter to quit): 
Goodbye!


## Key Takeaways (Recap for Students)| Concept | What We Did ||---|---|| API keys | Read from Colab Secrets or a hidden `getpass` box — never hardcoded || System prompt | Told the LLM to answer *only* from context and admit when it doesn't know || User prompt | Combined the retrieved chunks + the question, labeled with source names || Low temperature | `temperature=0.1` keeps answers factual and consistent || Groq | `groq` SDK, `chat.completions.create(...)`, fast open models || Gemini | `google-genai` SDK, `models.generate_content(...)` with `system_instruction` || Model fallback | Tried a list of models so a renamed/unavailable model doesn't crash the notebook || Threshold guard | Skipped the LLM entirely when nothing relevant was retrieved |**The complete RAG pipeline:** Load → Chunk → Embed → Store (Steps 1-2) → Retrieve (Step 3) →Generate a grounded answer (Step 4).### Practice Exercises1. Add your own `.txt` content as a third document in the `documents` dictionary and ask questions about it.2. Change `temperature` from `0.1` to `1.0` and compare how the answers change.3. Change the system prompt so the answer is returned as exactly **3 bullet points**.4. Set `n_results=1` in `retrieve_context()` and test Test 2 again — what information is lost?

## 10. Practice Exercise 1 — Add a Third Document

We add a small `.txt`-style document to the existing `documents` dictionary, rebuild the chunks and vector collection, and then ask questions that depend on the new content.

In [15]:
# Third document added for Practice Exercise 1
documents["rag_best_practices.txt"] = """
RAG Best Practices
A reliable RAG system should retrieve relevant chunks before asking an LLM to answer.
Similarity thresholds help reject weak matches and reduce hallucinations.
Hybrid search can combine semantic retrieval with keyword matching when exact terms,
IDs, or product codes matter.
The final answer should be grounded only in the retrieved context and should cite
the source documents used.
"""


# ---------------------------------------------------------
# Initialize ChromaDB client
# ---------------------------------------------------------
import chromadb

chroma_client = chromadb.Client()


# ---------------------------------------------------------
# Make sure the embedding function exists
# ---------------------------------------------------------
# If embedding_fn was already created in an earlier cell,
# this will use it.
#
# Otherwise, create it using Sentence Transformers.

try:
    embedding_fn
except NameError:
    from chromadb.utils import embedding_functions

    embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )


# ---------------------------------------------------------
# Rebuild chunks
# ---------------------------------------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

all_chunks = []
all_metadata = []

for source, text in documents.items():
    chunks = text_splitter.split_text(text)

    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)

        all_metadata.append({
            "source": source,
            "chunk_index": i
        })


# ---------------------------------------------------------
# Create a fresh Chroma collection
# ---------------------------------------------------------
collection = chroma_client.get_or_create_collection(
    name="rag_step4_with_third_doc",
    embedding_function=embedding_fn
)


# ---------------------------------------------------------
# Reset collection if it already contains data
# ---------------------------------------------------------
existing = collection.count()

if existing:
    collection.delete(where={})


# ---------------------------------------------------------
# Add all chunks to ChromaDB
# ---------------------------------------------------------
collection.add(
    ids=[f"chunk_{i}" for i in range(len(all_chunks))],
    documents=all_chunks,
    metadatas=all_metadata
)


# ---------------------------------------------------------
# Display collection information
# ---------------------------------------------------------
print("Documents:", list(documents.keys()))
print("Total chunks:", collection.count())


# ---------------------------------------------------------
# Practice Exercise 1
# ---------------------------------------------------------
exercise1_query = "How do similarity thresholds help reduce hallucinations in RAG?"

exercise1_chunks = retrieve_context(exercise1_query)


print("\nQuestion:", exercise1_query)
print("Retrieved chunks:", len(exercise1_chunks))

for c in exercise1_chunks:
    print(
        f"[{c['similarity']}] "
        f"{c['source']} #{c['chunk_index']}"
    )

Documents: ['ai_overview.txt', 'embeddings_in_practice.txt', 'rag_best_practices.txt']
Total chunks: 8

Question: How do similarity thresholds help reduce hallucinations in RAG?
Retrieved chunks: 2
[0.4312] rag_best_practices.txt #0
[0.3582] ai_overview.txt #3


In [16]:
# Ask the LLM about the newly added document.
print(rag_answer(
    "What are two best practices for making a RAG system more reliable?",
    provider=default_provider
))


[Gemini skipped: no API key provided]

Retrieved from: ai_overview.txt #3, rag_best_practices.txt #0


## 11. Practice Exercise 2 — Compare Temperature 0.1 vs 1.0

Temperature controls how varied the model's generation can be. We keep the same retrieved context and question so that only the temperature changes.

In [17]:
def ask_groq_with_temperature(user_prompt, temperature=0.1):
    """Call Groq using the supplied temperature."""
    if not GROQ_API_KEY:
        return "[Groq skipped: no API key provided]"

    client = Groq(api_key=GROQ_API_KEY)
    last_error = None

    for model_name in GROQ_MODELS:
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=temperature,
                max_tokens=600
            )
            if response.choices and response.choices[0].message.content:
                return response.choices[0].message.content.strip()
        except Exception as e:
            last_error = e

    return f"[Groq error: {last_error}]"


def ask_gemini_with_temperature(user_prompt, temperature=0.1):
    """Call Gemini using the supplied temperature."""
    if not GEMINI_API_KEY:
        return "[Gemini skipped: no API key provided]"

    client = genai.Client(api_key=GEMINI_API_KEY)
    last_error = None

    for model_name in GEMINI_MODELS:
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    temperature=temperature,
                    max_output_tokens=600
                )
            )
            if response.text:
                return response.text.strip()
        except Exception as e:
            last_error = e

    return f"[Gemini error: {last_error}]"


temperature_question = "Why is RAG useful for reducing unsupported answers?"
temperature_chunks = retrieve_context(temperature_question)
temperature_prompt = build_user_prompt(
    temperature_question,
    temperature_chunks
)

print("QUESTION:", temperature_question)

if GROQ_API_KEY:
    print("\n" + "=" * 20 + " GROQ: temperature 0.1 " + "=" * 20)
    print(ask_groq_with_temperature(temperature_prompt, 0.1))
    print("\n" + "=" * 20 + " GROQ: temperature 1.0 " + "=" * 20)
    print(ask_groq_with_temperature(temperature_prompt, 1.0))
elif GEMINI_API_KEY:
    print("\n" + "=" * 20 + " GEMINI: temperature 0.1 " + "=" * 20)
    print(ask_gemini_with_temperature(temperature_prompt, 0.1))
    print("\n" + "=" * 20 + " GEMINI: temperature 1.0 " + "=" * 20)
    print(ask_gemini_with_temperature(temperature_prompt, 1.0))
else:
    print("No API key is available, so the temperature comparison is skipped.")


QUESTION: Why is RAG useful for reducing unsupported answers?
No API key is available, so the temperature comparison is skipped.


## 12. Practice Exercise 3 — Force Exactly 3 Bullet Points

The system prompt is changed so the model must return exactly three bullet points and must still refuse to answer when the information is not present in the retrieved context.

In [18]:
THREE_BULLET_SYSTEM_PROMPT = (
    "You are a helpful RAG assistant. Answer ONLY using the provided context. "
    "If the answer is not contained in the context, reply exactly: "
    '"I don\'t know based on the provided documents." '
    "Otherwise, return exactly 3 bullet points. "
    "Each bullet point must be concise and factual. "
    "Do not add an introduction or conclusion. "
    "After the 3 bullet points, include the source file name(s) used."
)

def build_three_bullet_prompt(query, context_chunks):
    context_text = "\n\n".join(
        f"[Source: {c['source']}, chunk {c['chunk_index']}]\n{c['text']}"
        for c in context_chunks
    )
    return (
        f"Context:\n{context_text}\n\n"
        f"Question: {query}\n\n"
        "Return exactly 3 bullet points:"
    )

def ask_groq_three_bullets(user_prompt):
    if not GROQ_API_KEY:
        return "[Groq skipped: no API key provided]"
    client = Groq(api_key=GROQ_API_KEY)
    last_error = None
    for model_name in GROQ_MODELS:
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": THREE_BULLET_SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.1,
                max_tokens=500
            )
            if response.choices and response.choices[0].message.content:
                return response.choices[0].message.content.strip()
        except Exception as e:
            last_error = e
    return f"[Groq error: {last_error}]"

def ask_gemini_three_bullets(user_prompt):
    if not GEMINI_API_KEY:
        return "[Gemini skipped: no API key provided]"
    client = genai.Client(api_key=GEMINI_API_KEY)
    last_error = None
    for model_name in GEMINI_MODELS:
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=THREE_BULLET_SYSTEM_PROMPT,
                    temperature=0.1,
                    max_output_tokens=500
                )
            )
            if response.text:
                return response.text.strip()
        except Exception as e:
            last_error = e
    return f"[Gemini error: {last_error}]"

three_bullet_question = "What are the main benefits of RAG?"
three_bullet_chunks = retrieve_context(three_bullet_question)
three_bullet_prompt = build_three_bullet_prompt(
    three_bullet_question, three_bullet_chunks
)

if GROQ_API_KEY:
    print(ask_groq_three_bullets(three_bullet_prompt))
elif GEMINI_API_KEY:
    print(ask_gemini_three_bullets(three_bullet_prompt))
else:
    print("No API key is available, so the 3-bullet-point generation is skipped.")


No API key is available, so the 3-bullet-point generation is skipped.


## 13. Practice Exercise 4 — Set `n_results=1`

Retrieving only one chunk reduces the amount of context sent to the LLM. For a question that needs information from multiple documents, this can cause important evidence to be lost.

In [19]:
q4 = "How do embeddings and vector databases help RAG find useful information?"

# Original setting: up to 4 relevant chunks
chunks_4 = retrieve_context(q4, n_results=4)

# Exercise setting: only the single best chunk
chunks_1 = retrieve_context(q4, n_results=1)

print("Question:", q4)

print("\n--- n_results=4 ---")
for c in chunks_4:
    print(f"[{c['similarity']}] {c['source']} #{c['chunk_index']}: {c['text'][:180]}...")

print("\n--- n_results=1 ---")
for c in chunks_1:
    print(f"[{c['similarity']}] {c['source']} #{c['chunk_index']}: {c['text'][:180]}...")

print("\nObservation:")
if len(chunks_4) > len(chunks_1):
    print("Using n_results=1 removes additional retrieved context, which can lose information needed for multi-part questions.")
else:
    print("The threshold/available results produced the same number of usable chunks in this run.")


Question: How do embeddings and vector databases help RAG find useful information?

--- n_results=4 ---
[0.7577] embeddings_in_practice.txt #0: Embeddings and Vector Search in Practice...
[0.6304] embeddings_in_practice.txt #2: Vector databases such as ChromaDB store these embeddings and allow fast similarity search across thousands or millions of documents at once....
[0.5609] embeddings_in_practice.txt #1: Modern search engines increasingly rely on embeddings rather than simple keyword matching. An embedding model converts a search query into the same vector space as the stored docum...
[0.4405] ai_overview.txt #3: Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a language model. Instead of relying only on what a language model memorized during traini...

--- n_results=1 ---
[0.7577] embeddings_in_practice.txt #0: Embeddings and Vector Search in Practice...

Observation:
Using n_results=1 removes additional retrieved context, which can lose i